In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import scipy.io as sio
import spikeinterface.extractors as se
import spikeinterface as si
import spikeinterface.sorters as ss
import spikeinterface.postprocessing as spost
import spikeinterface.qualitymetrics as sqm
from pathlib import Path
import matplotlib.pyplot as plt
import json
from typing import List, Tuple
from scipy.stats import pearsonr
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT_SORT_DIR = "/media/ubuntu/sda/duan/result/rep_1/phy_folder_for_kilosort"

# 需要的 cluster 指标文件（来自 phy_folder_for_kilosort）
CLUSTER_INFO_FILENAME = "cluster_info.tsv"

SPIKE_CLUSTERS_FILENAME = "spike_clusters.npy"
SPIKE_TIMES_FILENAME = "spike_times.npy"


def load_cluster_info(phy_dir: str) -> pd.DataFrame:
    """读取 phy_folder_for_kilosort/cluster_info.tsv 为 DataFrame。
    该表包含所有需要的 cluster 级指标。
    如果cluster_info.tsv不存在，则尝试从其他文件构建基本信息。
    同时计算并添加mean_waveform信息。
    """
    path = os.path.join(phy_dir, CLUSTER_INFO_FILENAME)
    
    if os.path.exists(path):
        df = pd.read_csv(path, sep='\t')
        # 标准化主键列名
        if 'cluster_id' not in df.columns:
            raise ValueError(f"{path} 中缺少 cluster_id 列")
    else:
        print(f"警告: {path} 不存在，尝试从其他文件构建cluster信息")
        
        # 尝试从cluster_group.tsv构建基本信息
        cluster_group_path = os.path.join(phy_dir, "cluster_group.tsv")
        if os.path.exists(cluster_group_path):
            df = pd.read_csv(cluster_group_path, sep='\t')
            if 'cluster_id' not in df.columns:
                raise ValueError(f"{cluster_group_path} 中缺少 cluster_id 列")
        else:
            # 如果都没有，从spike_clusters.npy中提取唯一的cluster_id
            spike_clusters_path = os.path.join(phy_dir, SPIKE_CLUSTERS_FILENAME)
            if os.path.exists(spike_clusters_path):
                spike_clusters = np.load(spike_clusters_path)
                unique_clusters = np.unique(spike_clusters)
                df = pd.DataFrame({
                    'cluster_id': unique_clusters,
                    'group': 'unsorted'  # 默认分组
                })
            else:
                raise ValueError(f"无法找到任何cluster信息文件: {phy_dir}")
    
    # 计算mean_waveform并展开为多列添加到DataFrame
    cluster_waveform_data = compute_mean_waveform(phy_dir)
    if cluster_waveform_data:
        # 确定waveform的长度（通常是90个时间点）
        waveform_length = None
        for cluster_data in cluster_waveform_data.values():
            if cluster_data['waveform'] is not None:
                waveform_length = len(cluster_data['waveform'])
                break
        
        if waveform_length is not None:
            # 添加位置信息和best_channels列
            df['position_1'] = np.nan
            df['position_2'] = np.nan
            df['best_channels'] = None
            
            # 创建waveform列名
            waveform_columns = [f'mean_waveform_{i}' for i in range(waveform_length)]
            
            # 为每个cluster创建waveform数据
            waveform_data = {}
            for cluster_id in df['cluster_id']:
                if cluster_id in cluster_waveform_data:
                    cluster_data = cluster_waveform_data[cluster_id]
                    
                    # 添加位置信息和best_channels
                    pos_x, pos_y = cluster_data['position']
                    df.loc[df['cluster_id'] == cluster_id, 'position_1'] = pos_x
                    df.loc[df['cluster_id'] == cluster_id, 'position_2'] = pos_y
                    df.loc[df['cluster_id'] == cluster_id, 'best_channels'] = str(cluster_data['channels'])
                    
                    waveform = cluster_data['waveform']
                    # 确保waveform长度一致
                    if waveform is not None and len(waveform) > 0:
                        if len(waveform) == waveform_length:
                            waveform_data[cluster_id] = waveform
                        else:
                            # 如果长度不匹配，用NaN填充或截断
                            padded_waveform = np.full(waveform_length, np.nan)
                            padded_waveform[:min(len(waveform), waveform_length)] = waveform[:min(len(waveform), waveform_length)]
                            waveform_data[cluster_id] = padded_waveform
                    else:
                        waveform_data[cluster_id] = np.full(waveform_length, np.nan)
                else:
                    # 没有数据的cluster用NaN填充
                    waveform_data[cluster_id] = np.full(waveform_length, np.nan)
            
            # 将waveform数据转换为DataFrame并合并
            waveform_df = pd.DataFrame.from_dict(waveform_data, orient='index', columns=waveform_columns)
            waveform_df.index.name = 'cluster_id'
            waveform_df = waveform_df.reset_index()
            
            df = df.merge(waveform_df, on='cluster_id', how='left')
        else:
            print("警告: 无法确定waveform长度")
    else:
        print("警告: 无法计算cluster数据")
        # 添加空的位置列和best_channels列
        df['position_1'] = np.nan
        df['position_2'] = np.nan
        df['best_channels'] = None
    
    return df


def load_spike_level(phy_dir: str) -> pd.DataFrame:
    """读取 spike 层面的 numpy 文件并返回 DataFrame: [cluster, time]
    time 使用原始采样点，不做单位转换。
    """
    spike_clusters = np.load(os.path.join(phy_dir, SPIKE_CLUSTERS_FILENAME))
    spike_times = np.load(os.path.join(phy_dir, SPIKE_TIMES_FILENAME))
    # 展平为一维
    spike_clusters = np.asarray(spike_clusters).reshape(-1)
    spike_times = np.asarray(spike_times).reshape(-1)
    if spike_clusters.shape[0] != spike_times.shape[0]:
        raise ValueError(f"spike_clusters 与 spike_times 行数不一致: {phy_dir}")
    df = pd.DataFrame({
        'cluster_id': spike_clusters.astype(int),
        'time': spike_times.astype(int),
    })
    return df


def compute_mean_waveform(phy_dir: str) -> dict:
    """根据probe信息、template_ind.npy和templates.npy计算每个cluster的位置和波形
    
    Returns:
        dict: {cluster_id: {'position': (x, y), 'waveform': waveform_array}}
    """
    templates_path = os.path.join(phy_dir, "templates.npy")
    template_ind_path = os.path.join(phy_dir, "template_ind.npy")
    
    if not all(os.path.exists(p) for p in [templates_path, template_ind_path]):
        print("警告: 缺少templates.npy或template_ind.npy文件")
        return {}
    
    # 加载数据
    templates = np.load(templates_path)  # shape: (n_templates, n_timepoints, n_channels)
    template_ind = np.load(template_ind_path)  # shape: (n_templates, n_channels)
    
    print(f"Templates shape: {templates.shape}")
    print(f"Template_ind shape: {template_ind.shape}")
    
    # 使用全局probe信息
    global probe
    
    # 获取probe的通道位置信息
    channel_positions = {}
    for i, contact_id in enumerate(probe.contact_ids):
        x, y = probe.contact_positions[i]
        # 处理浮点数contact_id
        channel_id = int(float(contact_id))
        channel_positions[channel_id] = [x, y]
    
    cluster_results = {}
    
    # 处理每个cluster
    for cluster_id in range(templates.shape[0]):
        template = templates[cluster_id]  # shape: (n_timepoints, n_channels)
        template_channels = template_ind[cluster_id]  # shape: (n_channels,)
        
        # 过滤掉-1的通道
        valid_channels = template_channels[template_channels != -1]
        if len(valid_channels) == 0:
            continue
            
        # 获取有效通道的波形数据
        valid_template = template[:, template_channels != -1]  # shape: (n_timepoints, n_valid_channels)
        
        # 计算每个通道的波形幅度（使用RMS）
        channel_amplitudes = np.sqrt(np.mean(valid_template**2, axis=0))
        
        # 根据通道位置和波形幅度计算cluster位置
        total_amplitude = 0
        weighted_x = 0
        weighted_y = 0
        
        for i, channel_id in enumerate(valid_channels):
            channel_id_int = int(float(channel_id))
            if channel_id_int in channel_positions:
                x, y = channel_positions[channel_id_int]
                amplitude = channel_amplitudes[i]
                
                weighted_x += x * amplitude
                weighted_y += y * amplitude
                total_amplitude += amplitude
        
        if total_amplitude > 0:
            cluster_x = weighted_x / total_amplitude
            cluster_y = weighted_y / total_amplitude
        else:
            cluster_x, cluster_y = 0, 0
        
        # 根据cluster位置反推波形大小
        # 计算到每个通道的距离
        distances = []
        for channel_id in valid_channels:
            channel_id_int = int(float(channel_id))
            if channel_id_int in channel_positions:
                x, y = channel_positions[channel_id_int]
                distance = np.sqrt((cluster_x - x)**2 + (cluster_y - y)**2)
                distances.append(distance)
            else:
                distances.append(float('inf'))
        
        # 使用IDW (Inverse Distance Weighting) 计算合成波形
        if len(distances) > 0 and not all(d == float('inf') for d in distances):
            # 避免除零
            distances = np.array(distances)
            distances[distances == 0] = 1e-6
            
            # 计算权重 (power=2)
            weights = 1 / (distances ** 2)
            weights = weights / np.sum(weights)
            
            # 合成波形
            synthesized_waveform = np.zeros(valid_template.shape[0])
            for t in range(valid_template.shape[0]):
                synthesized_waveform[t] = np.dot(valid_template[t, :], weights)
        else:
            # 如果无法计算距离，使用平均波形
            synthesized_waveform = np.mean(valid_template, axis=1)
        
        cluster_results[cluster_id] = {
            'position': (cluster_x, cluster_y),
            'waveform': synthesized_waveform,
            'channels': valid_channels.tolist(),
            'amplitudes': channel_amplitudes.tolist()
        }
    
    print(f"成功处理了 {len(cluster_results)} 个clusters")
    return cluster_results

In [3]:
probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y

probe = Probe()
probe.set_contacts(positions=probe_position, contact_ids=probe_data['chanMap'][:, 0])

probe_loc = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
probe.set_device_channel_indices(probe_loc['probeloc'].values)

In [4]:
cluster_inf = load_cluster_info(phy_dir=ROOT_SORT_DIR)
spike_inf = load_spike_level(phy_dir=ROOT_SORT_DIR)

cluster_inf = cluster_inf[cluster_inf['group'] == 'good']
spike_inf = spike_inf[spike_inf['cluster_id'].isin(cluster_inf['cluster_id'].values)]

Templates shape: (148, 90, 6)
Template_ind shape: (148, 6)
成功处理了 148 个clusters


In [5]:
cluster_inf.to_csv('cluster_inf.csv')
spike_inf.to_csv('spike_inf.tsv', sep = '\t')